### ***727723EUAI109 : Sanjay M***

### ***Skill Builder : 2***

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, Dense

reviews = [
    "The application provides stable performance during daily operations",
    "There are multiple issues affecting the overall system reliability",
    "User interface is intuitive and easy to understand",
    "The response time is slower than expected under peak load",
    "System updates improved performance and fixed previous issues",
    "The documentation is unclear and lacks sufficient details",
    "The platform maintains consistent functionality across sessions",
    "There are occasional failures during transaction processing",
    "The design is structured and follows standard guidelines",
    "Performance degradation is observed during extended usage"
]

MAXLEN = 100
EMBED_DIM = 16

selected_review = reviews[0]
words = selected_review.lower().split()[:MAXLEN]
review_text = " ".join(words)

tokenizer = Tokenizer()
tokenizer.fit_on_texts([review_text])
sequence = tokenizer.texts_to_sequences([review_text])
padded = pad_sequences(sequence, maxlen=MAXLEN, padding='post')

vocab_size = len(tokenizer.word_index) + 1

embedding_layer = Embedding(input_dim=vocab_size, output_dim=EMBED_DIM)
embedded = embedding_layer(tf.constant(padded))  # shape: (1, MAXLEN, EMBED_DIM)

dense_layer = Dense(1)
scores = dense_layer(embedded)                   
scores = tf.squeeze(scores, axis=-1)             

mask = tf.cast(tf.not_equal(padded, 0), dtype=tf.float32)  # (1, MAXLEN)
scores = scores * mask + (1 - mask) * -1e9

attention_weights = tf.nn.softmax(scores, axis=-1).numpy()[0]  # (MAXLEN,)

index_to_word = {v: k for k, v in tokenizer.word_index.items()}
print(f"Review: {review_text}\n")
print(f"{'Word':<20} {'Attention Weight'}")
print("-" * 38)
for idx, weight in zip(padded[0], attention_weights):
    if idx == 0:
        break
    print(f"{index_to_word[idx]:<20} {weight:.6f}")

Review: the application provides stable performance during daily operations

Word                 Attention Weight
--------------------------------------
the                  0.126584
application          0.127904
provides             0.127363
stable               0.125873
performance          0.122283
during               0.132535
daily                0.114018
operations           0.123440
